In [33]:
import pandas as pd
import os
import unicodedata
import re
import sys
print(sys.executable)
from rapidfuzz import process, fuzz

c:\Users\Emre Cem\AppData\Local\Programs\Python\Python311\python.exe


Cleaning the results column for unplayed matches and convert date column into datetime object

In [34]:
# Load the processed all_matches data
team_centric_matches = pd.read_csv("../data/processed/team_centric_matches.csv")
team_centric_matches = team_centric_matches.dropna(subset=['result']).reset_index(drop=True)

# Sort by target team, season, and date
team_centric_matches = team_centric_matches.sort_values(by=['target_team', 'season', 'date']).reset_index(drop=True)

# Drop rows with missing 'result' values
team_centric_matches = team_centric_matches.dropna(subset=['result']).reset_index(drop=True)

# Convert 'date' column to datetime format
team_centric_matches['date'] = pd.to_datetime(team_centric_matches['date'], dayfirst=True)

# Verification
print(team_centric_matches.tail())

        target_team        opponent_team  is_home                date  \
27105  Ümraniyespor  Istanbul Basaksehir    False 2023-05-17 17:00:00   
27106  Ümraniyespor           Ankaragücü     True 2023-05-21 16:00:00   
27107  Ümraniyespor            Hatayspor    False 2023-05-30 00:00:00   
27108  Ümraniyespor          Giresunspor     True 2023-06-03 19:00:00   
27109  Ümraniyespor         Istanbulspor    False 2023-06-07 20:00:00   

      competition stage     season  days_since_last_ucl_match  \
27105   Super_Lig    34  2022-2023                        NaN   
27106   Super_Lig    35  2022-2023                        NaN   
27107   Super_Lig    36  2022-2023                        NaN   
27108   Super_Lig    37  2022-2023                        NaN   
27109   Super_Lig    38  2022-2023                        NaN   

       ucl_impact_category result  target_team_goals  opponent_team_goals  \
27105                  NaN  1 - 1                NaN                  NaN   
27106           

C:\Users\Emre Cem\AppData\Local\Temp\ipykernel_11320\3806029055.py:12: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  team_centric_matches['date'] = pd.to_datetime(team_centric_matches['date'], dayfirst=True)


Filling "target_team_goals" and "opponent_team_goals" columns

In [35]:
# Split the 'result' column into 'target_team_goals' and 'opponent_team_goals'
score_parts = team_centric_matches['result'].str.split(' - ', expand=True)

# Convert the split parts to integers and assign them to new columns
team_centric_matches['target_team_goals'] = score_parts[0].astype(int)
team_centric_matches['opponent_team_goals'] = score_parts[1].astype(int)

# Verification
print(team_centric_matches.head())

             target_team     opponent_team  is_home                date  \
0  1. FC Heidenheim 1846         Wolfsburg    False 2023-08-19 13:30:00   
1  1. FC Heidenheim 1846    TSG Hoffenheim     True 2023-08-26 13:30:00   
2  1. FC Heidenheim 1846          Dortmund    False 2023-09-01 18:30:00   
3  1. FC Heidenheim 1846  SV Werder Bremen     True 2023-09-17 13:30:00   
4  1. FC Heidenheim 1846        Leverkusen    False 2023-09-24 13:30:00   

  competition stage     season  days_since_last_ucl_match  \
0  Bundesliga     1  2023-2024                        NaN   
1  Bundesliga     2  2023-2024                        NaN   
2  Bundesliga     3  2023-2024                        NaN   
3  Bundesliga     4  2023-2024                        NaN   
4  Bundesliga     5  2023-2024                        NaN   

   ucl_impact_category result  target_team_goals  opponent_team_goals  points  \
0                  NaN  2 - 0                  2                    0     NaN   
1                  N

Calculating match counts. Filling the "target_team_match_count" and "oponent_team_match_count"

In [36]:
# Calculate the cumulative match count for each team
team_centric_matches['target_team_match_count'] = team_centric_matches.groupby(['target_team', 'season']).cumcount() + 1

# Calculate the cumulative match count for the opponent team
# to find it we need a lookup table
# this table maps a team's name and date to their specific match number
match_number_lookup = team_centric_matches[['target_team', 'date', 'target_team_match_count']].copy()

# rename the columns to match the 'opponent' perspective
match_number_lookup.columns = ['opponent_team', 'date', 'opponent_team_match_count']

# merge the lookup table back to the main DataFrame
# we drop the existing 'opponent_team_match_count' column to avoid duplicates
if 'opponent_team_match_count' in team_centric_matches.columns:
    team_centric_matches = team_centric_matches.drop(columns=['opponent_team_match_count'])

team_centric_matches = pd.merge(
    team_centric_matches, 
    match_number_lookup, 
    on=['opponent_team', 'date'], 
    how='left'
)

# verification  
print(team_centric_matches[team_centric_matches['target_team'] == 'Galatasaray'].tail(6))

       target_team   opponent_team  is_home                date  \
10244  Galatasaray       Liverpool    False 2026-03-18 20:00:00   
10245  Galatasaray     Trabzonspor    False 2026-04-04 20:00:00   
10246  Galatasaray         Göztepe    False 2026-04-08 20:00:00   
10247  Galatasaray     Kocaelispor     True 2026-04-12 20:00:00   
10248  Galatasaray  Gençlerbirligi    False 2026-04-18 20:00:00   
10249  Galatasaray      Fenerbahçe     True 2026-04-26 20:00:00   

            competition       stage     season  days_since_last_ucl_match  \
10244  Champions_League  R16 Game 2  2025-2026                        NaN   
10245         Super_Lig          28  2025-2026                        NaN   
10246         Super_Lig          27  2025-2026                        NaN   
10247         Super_Lig          29  2025-2026                        NaN   
10248         Super_Lig          30  2025-2026                        NaN   
10249         Super_Lig          31  2025-2026                      

Calculate points and determine UCL format

In [37]:
# 1: Calculate points
# Define a function to determine points from the target team's perspective
def determine_match_points(row):
    if row['target_team_goals'] > row['opponent_team_goals']:
        return 3
    elif row['target_team_goals'] == row['opponent_team_goals']:
        return 1
    else:
        return 0

# Apply the function to calculate points for each match 
# I didn't care about the UCL points because they are not relevant and some matches don't even have them (qualifiers, playoffs, QF, SF, F)
team_centric_matches['points'] = team_centric_matches.apply(determine_match_points, axis=1)


# 2: Determine the UCL format
# define the seasons with the new format
new_format_seasons = ['2024-2025', '2025-2026']

# Initialize the column with NA to ensure demostic league matches remain empty
team_centric_matches['ucl_format'] = pd.NA

# create a mask to only target UCL matches
ucl_matches_mask = team_centric_matches['competition'] == 'Champions_League'

# For the matches that are in the UCL, determine if they are in the new or old format based on the season
team_centric_matches.loc[ucl_matches_mask, 'ucl_format'] = team_centric_matches.loc[ucl_matches_mask, 'season'].apply(
    lambda x: 'new' if x in new_format_seasons else 'old'
)

# Verification
print(team_centric_matches['ucl_format'].value_counts(dropna=False))

ucl_format
<NA>    25360
old      1000
new       750
Name: count, dtype: int64


Calculate how many days have past since the last UCL match 

In [38]:
# 1: create a subset containing only UCL matches
# this will serve as a reference point for previous UCL dates
ucl_only_matches = team_centric_matches[team_centric_matches['competition'] == 'Champions_League'].copy()

# 2: select only the necessary columns for the lookup and rename the date column
ucl_dates_lookup = ucl_only_matches[['target_team', 'date']].copy()
ucl_dates_lookup.columns = ['target_team', 'last_ucl_date']

# 3: sort both the DataFrames by date for merge
team_centric_matches = team_centric_matches.sort_values('date')
ucl_dates_lookup = ucl_dates_lookup.sort_values('last_ucl_date')

# 4: Use merge_asof to find the closest previous UCL match for each row
team_centric_matches = pd.merge_asof(
    team_centric_matches,
    ucl_dates_lookup,
    left_on='date',
    right_on='last_ucl_date',
    by='target_team',
    direction='backward',
    allow_exact_matches=False
)

# 5: Calculate the days since the last UCL match
team_centric_matches['days_since_last_ucl_match'] = (
    team_centric_matches['date'] - team_centric_matches['last_ucl_date']
).dt.days
# 6: categorize the impact based on the days since the last UCL match
def categorize_ucl_impact(days):
    if pd.isna(days) or days > 6:
        return 'no_effect'
    elif days <= 3:
        return 'severe'
    else:  # 4-6 days
        return 'moderate'
    
team_centric_matches['ucl_impact_category'] = team_centric_matches['days_since_last_ucl_match'].apply(categorize_ucl_impact)

# verification
team_centric_matches = team_centric_matches.sort_values(['target_team', 'date']).reset_index(drop=True)
print(team_centric_matches[team_centric_matches['target_team'] == 'Galatasaray'].tail(11))

       target_team        opponent_team  is_home                date  \
10239  Galatasaray             Juventus    False 2026-02-25 20:00:00   
10240  Galatasaray           Alanyaspor     True 2026-02-28 20:00:00   
10241  Galatasaray             Besiktas    False 2026-03-07 20:00:00   
10242  Galatasaray            Liverpool     True 2026-03-10 17:45:00   
10243  Galatasaray  Istanbul Basaksehir     True 2026-03-14 20:00:00   
10244  Galatasaray            Liverpool    False 2026-03-18 20:00:00   
10245  Galatasaray          Trabzonspor    False 2026-04-04 20:00:00   
10246  Galatasaray              Göztepe    False 2026-04-08 20:00:00   
10247  Galatasaray          Kocaelispor     True 2026-04-12 20:00:00   
10248  Galatasaray       Gençlerbirligi    False 2026-04-18 20:00:00   
10249  Galatasaray           Fenerbahçe     True 2026-04-26 20:00:00   

            competition            stage     season  \
10239  Champions_League  Play-off Game 2  2025-2026   
10240         Super_Lig  

Pruning the Dataset

In [39]:
# 1: identify all unique teams that have participated in the UCL during the study period
all_matches = pd.read_csv("../data/processed/all_matches_raw_normalized.csv")
ucl_participants = team_centric_matches[team_centric_matches['competition'] == 'Champions_League']['target_team'].unique()

# 2: filter the dataset:
# keep a match if the target team is a UCL participant, this keeps all their league matches and their UCL matches
team_centric_matches = team_centric_matches[team_centric_matches['target_team'].isin(ucl_participants)].reset_index(drop=True)

relevant_matches_mask = (all_matches['target_team'].isin(ucl_participants)) | \
                        (all_matches['opponent_team'].isin(ucl_participants))

kept_matches_count = len(all_matches[relevant_matches_mask])
total_original_matches = len(all_matches)

# verification and impact assessment
print(f"Original match count: {total_original_matches}")
print(f"Number of matches containing a UCL team (filtered): {kept_matches_count}")
print(f"%{ (kept_matches_count / total_original_matches) * 100:.2f} of the dataset was retained for analysis.")

print(team_centric_matches[team_centric_matches['target_team'] == 'Galatasaray'].tail(11))

Original match count: 13781
Number of matches containing a UCL team (filtered): 8135
%59.03 of the dataset was retained for analysis.
      target_team        opponent_team  is_home                date  \
3702  Galatasaray             Juventus    False 2026-02-25 20:00:00   
3703  Galatasaray           Alanyaspor     True 2026-02-28 20:00:00   
3704  Galatasaray             Besiktas    False 2026-03-07 20:00:00   
3705  Galatasaray            Liverpool     True 2026-03-10 17:45:00   
3706  Galatasaray  Istanbul Basaksehir     True 2026-03-14 20:00:00   
3707  Galatasaray            Liverpool    False 2026-03-18 20:00:00   
3708  Galatasaray          Trabzonspor    False 2026-04-04 20:00:00   
3709  Galatasaray              Göztepe    False 2026-04-08 20:00:00   
3710  Galatasaray          Kocaelispor     True 2026-04-12 20:00:00   
3711  Galatasaray       Gençlerbirligi    False 2026-04-18 20:00:00   
3712  Galatasaray           Fenerbahçe     True 2026-04-26 20:00:00   

           co